# Self-Distillation Early-Exit — Training (4 backends, grid + sync)

Trains the chosen backends end-to-end, then per-iteration:
- pushes checkpoints (`*.pt`, adapter files) to a private HF Hub repo (one per grid item),
- mirrors `metrics.json` logs to a Google Drive folder (cheap, fits 15 GB free).

Killing the cell mid-grid leaves earlier items fully synced. Resumption is automatic via `metrics.json` markers — re-running the cell skips finished stages.


## 1. Secrets (Colab → Secrets or `.env`)


In [ ]:
import os

def _secret(key):
    try:
        from google.colab import userdata
        return userdata.get(key)
    except Exception:
        return os.environ.get(key)

for _k in ("HF_TOKEN", "HF_USER", "GH_TOKEN"):
    _v = _secret(_k)
    if _v: os.environ[_k] = _v

assert os.environ.get("HF_TOKEN"), "set HF_TOKEN in Colab Secrets — checkpoint push will use it"
assert os.environ.get("HF_USER"),  "set HF_USER  in Colab Secrets — used as repo owner"
print("HF_USER:", os.environ["HF_USER"])


## 2. Clone repo


In [ ]:
import os, subprocess

USER, REPO = "airlanggawicaksono", "EarlyExitMonoRepo"
_tok = os.environ.get("GH_TOKEN")
_url = f"https://{_tok}@github.com/{USER}/{REPO}.git" if _tok else f"https://github.com/{USER}/{REPO}.git"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", _url], check=True)
%cd {REPO}


## 3. Mount Drive (for logs)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 4. Drive-pinned caches (avoid re-download every session)
Pinning HF cache + dataset cache to Drive means model weights / tokenized datasets persist across Colab sessions. Set BEFORE any HF / datasets import so downloads land in the right place.


In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/selfdistill")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

HF_CACHE   = DRIVE_ROOT / "hf_cache"
DATA_ROOT  = DRIVE_ROOT / "data"           # GLUE TSVs, YOLO data yamls, etc
WEIGHTS    = DRIVE_ROOT / "weights"        # gelan-s.pt, custom ckpts
DRIVE_LOGS = DRIVE_ROOT / "logs"           # metrics.json mirror + run outputs
for p in (HF_CACHE, DATA_ROOT, WEIGHTS, DRIVE_LOGS):
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"]            = str(HF_CACHE)
os.environ["HF_DATASETS_CACHE"]  = str(HF_CACHE / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE / "transformers")
print("HF cache  ->", HF_CACHE)
print("data root ->", DATA_ROOT)
print("weights   ->", WEIGHTS)
print("logs      ->", DRIVE_LOGS)


## 5. Install deps
Restart runtime AFTER this cell (torchao removal needs it).


In [ ]:
!pip install -q peft datasets thop huggingface_hub
!pip uninstall -y torchao


## 6. HF login


In [ ]:
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HF login ok | user:", os.environ["HF_USER"])


## 7. Choose what to train
Toggle the flags; the grid below builds only the chosen items. Each item = one (backend, dataset, mode).


In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

RUN_BERT   = True
RUN_YOLO   = False     # needs gelan-s.pt + Roboflow / COCO data.yaml; see cell 7
RUN_VISION = False     # ViT on CIFAR / ImageNet
RUN_LLAMA  = False     # C4 streaming — long; flip on only for serious runs

BERT_TASKS    = ("SST-2", "MNLI")     # any GLUE task with prepared TSVs
VISION_DATASETS = ("uoft-cs/cifar10",)
YOLO_RUNS = ()                          # tuples of (label, data_yaml, weights)
LLAMA_MODES = ("joint",)
MODES = ("joint", "pairwise", "cascade")


## 8. (optional) Prepare data
GLUE TSVs for BERT. YOLO needs weights + data yaml — drop those into `weights/` and `data_*.yaml` before flipping `RUN_YOLO=True`.


In [ ]:
if RUN_BERT:
    from AnyTimeBert import prepare_task
    GLUE_DIR = DATA_ROOT / "glue_data"
    GLUE_DIR.mkdir(parents=True, exist_ok=True)
    for t in BERT_TASKS:
        prepare_task(t, out_root=GLUE_DIR)
    print("BERT data ready in Drive:", GLUE_DIR)


## 9. Build the grid


In [ ]:
from GeneralizableSelfDistilationEarlyExitTraining.runner import GridItem

items = []

if RUN_BERT:
    from GeneralizableSelfDistilationEarlyExitTraining.backends.bert import Cfg as BertCfg, train as bert_train
    for task in BERT_TASKS:
        for mode in MODES:
            items.append(GridItem(
                label=f"bert-{task.lower()}-{mode}",
                train_fn=bert_train,
                cfg=BertCfg(task=task, mode=mode, epochs=3, device=DEVICE, data_dir=GLUE_DIR, out_root=DRIVE_LOGS / "runs"),
            ))

if RUN_VISION:
    from GeneralizableSelfDistilationEarlyExitTraining.backends.vision import Cfg as VisCfg, train as vis_train
    for dset in VISION_DATASETS:
        for mode in MODES:
            slug = dset.split("/")[-1]
            items.append(GridItem(
                label=f"vision-{slug}-{mode}",
                train_fn=vis_train,
                cfg=VisCfg(dataset=dset, mode=mode, epochs=20, device=DEVICE, out_root=DRIVE_LOGS / "runs"),
            ))

if RUN_YOLO:
    from GeneralizableSelfDistilationEarlyExitTraining.backends.yolo import YoloCfg, train as yolo_train
    for ylabel, data_yaml, weights in YOLO_RUNS:
        for mode in MODES:
            items.append(GridItem(
                label=f"yolo-{ylabel}-{mode}",
                train_fn=yolo_train,
                cfg=YoloCfg(mode=mode, data_yaml=Path(data_yaml),
                            weights=Path(weights), epochs=50, device=DEVICE, out_root=DRIVE_LOGS / "runs"),
            ))

if RUN_LLAMA:
    from GeneralizableSelfDistilationEarlyExitTraining.backends.llama import Cfg as LlamaCfg, train as llama_train
    for mode in LLAMA_MODES:
        items.append(GridItem(
            label=f"llama-c4-{mode}",
            train_fn=llama_train,
            cfg=LlamaCfg(mode=mode, epochs=1, max_train_samples=100_000, device=DEVICE, out_root=DRIVE_LOGS / "runs"),
        ))

print(f"grid has {len(items)} items:")
for it in items: print(" ", it.label)


## 10. Run + sync
Each item: train → push ckpts to `<HF_USER>/selfdistill-<label>` (private) → copy `metrics.json` to Drive.


In [ ]:
from GeneralizableSelfDistilationEarlyExitTraining.runner import run_grid

run_grid(
    items,
    hf_user=os.environ["HF_USER"],
    hf_token=os.environ["HF_TOKEN"],
    drive_log_root=str(DRIVE_LOGS / "mirror"),
    repo_prefix="selfdistill",
    private=True,
)
print("\nall grid items done — checkpoints on HF, logs on Drive.")
